# 17 — Baseline Comparison and Proxy Metrics (headline evaluation)

Recommender vs baseline ladder, ceiling-normalized recall, and a proxy-metric vector, run interactively. The numbers reported in the thesis come from `run_osm_rome.py` and `run_geolife.py`, which run the same comparison under the final protocols; NB 16 is qualitative and NB 18 is diagnostic.

**Baselines**: random, global popularity, category-popularity, geographic proximity, Markov (1-step flow, leakage-free).

**Unified**: default weights + route-heavy configuration from NB 18's weight sweep.

**Metrics per method**:
- `R@k`, `R@k / ceiling`, `Δ vs popularity`, `Δ vs category_popularity`, `MRR`.
- `coverage@10`, `novelty@10`, `diversity@10`, `plausibility@10`, `spatial_cohesion@10`, `tag_match@10`.

Eligibility fixed at `min_seq_length=3`. `min_seq_length=5` is reported for reference (few trajectories, wide CIs).

Context: the product is the profile-card tag layer (NB 15). This notebook measures one downstream consumer — recommendation.

In [ ]:
import sys
import time
from functools import partial
from pathlib import Path

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from helpers.unified_recommender import UnifiedRecommender, UNIFIED_WEIGHTS
from helpers.baselines import (
    popularity_rec, category_popularity_rec, proximity_rec, markov_rec,
)
from helpers.baseline_evaluation import evaluate_method, lift_table

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
    'savefig.edgecolor': 'white',
    'axes.edgecolor': '#333333',
    'axes.labelcolor': '#111111',
    'text.color': '#111111',
    'xtick.color': '#333333',
    'ytick.color': '#333333',
})
sns.set_palette('husl')
%matplotlib inline

OUTPUT_DIR = Path('./output/evaluation')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Imports ready.')


In [ ]:
# Load unified recommender (provides data for all baselines too)
FEATURES_PATH = './output/semantic_features/trajectory_features.json'
t0 = time.time()
rec = UnifiedRecommender()
rec.load_all_data(features_path=FEATURES_PATH)
print(f'Load time: {time.time()-t0:.1f}s')
print(f'Trajectories: {len(rec.sequences)}  |  POIs: {len(rec.all_poi_uris)}')

In [ ]:
# Random ranker (local — not in baselines.py because it needs seeding per call)
def random_rec(rec_engine, traj_uri, input_seq, top_k, _rng=np.random.default_rng(42)):
    from helpers.context_recommender import EXCLUDED_CATEGORIES
    visited = set(input_seq)
    candidates = [p for p in rec_engine.all_poi_uris - visited
                  if rec_engine.poi_metadata.get(p, {}).get('category', '')
                  not in EXCLUDED_CATEGORIES]
    _rng.shuffle(candidates)
    out = []
    for p in candidates[:top_k]:
        m = rec_engine.poi_metadata.get(p, {})
        out.append({'poi': p, 'label': m.get('label', ''),
                    'category': m.get('category', ''),
                    'score': 0.0, 'signals': {}})
    return out


# Unified rankers using fixed weight configurations from notebook 18
ROUTE_HEAVY_WEIGHTS = {
    'archetype_sim': 0.10, 'semantic_sim': 0.10, 'route_sim': 0.30,
    'flow_prob': 0.20, 'dominant_ctx': 0.08, 'user_pref': 0.08,
    'significant': 0.05, 'engagement': 0.04, 'rule_coverage': 0.05,
}

def unified_rec(rec_engine, traj_uri, input_seq, top_k, weights=None):
    recs = rec_engine.recommend(
        traj_uri, observed_sequence=input_seq, top_k=top_k,
        signal_weights=weights, exclude_traj=traj_uri,
    )
    return recs

METHODS = {
    'random':              random_rec,
    'popularity':          popularity_rec,
    'category_popularity': category_popularity_rec,
    'proximity':           proximity_rec,
    'markov':              markov_rec,
    'unified_default':     partial(unified_rec, weights=UNIFIED_WEIGHTS),
    'unified_route_heavy': partial(unified_rec, weights=ROUTE_HEAVY_WEIGHTS),
}
print(f'{len(METHODS)} methods registered: {list(METHODS)}')

## Run evaluation at min_seq_length=3 (trajectory_split, ratio=0.7)

In [ ]:
results_mlen3 = {}
t_total = time.time()
for name, fn in METHODS.items():
    print(f'\n--- {name} ---')
    t0 = time.time()
    results_mlen3[name] = evaluate_method(
        rec, fn, method_name=name,
        k_values=(5, 10, 20),
        protocol='trajectory_split', split_ratio=0.7,
        min_seq_length=3, proxy_k=10,
        verbose=False,
    )
    s = results_mlen3[name]
    print(f'  N={s["n_evaluated"]}  R@10={s["recall@10"]:.4f}  '
          f'R@10/ceil={s["normalized_recall@10"]:.3f}  MRR={s["mrr"]:.4f}  '
          f'[{time.time()-t0:.1f}s]')
print(f'\nCeiling (min_seq=3): {results_mlen3["popularity"]["ceiling_recall"]:.3f}')
print(f'Total: {time.time()-t_total:.1f}s')

In [ ]:
# Comparison table (min_seq=3)
rows = lift_table(results_mlen3,
                  baseline_keys=('popularity', 'category_popularity'), k=10)
df3 = pd.DataFrame(rows)

display_cols = [
    'method', 'N', 'R@10', 'R@10/ceil',
    'Δ vs popularity', 'Δ vs category_popularity', 'MRR',
    'Cov@10', 'Nov@10', 'Div@10', 'Plaus@10', 'Cohes@10', 'TagMatch@10',
]
df3_view = df3[display_cols]
print('\nBaseline comparison, min_seq_length=3:')
display(df3_view.style.format({
    'R@10': '{:.4f}', 'R@10/ceil': '{:.3f}',
    'Δ vs popularity': '{:+.4f}', 'Δ vs category_popularity': '{:+.4f}',
    'MRR': '{:.4f}',
    'Cov@10': '{:.3f}', 'Nov@10': '{:.3f}', 'Div@10': '{:.3f}',
    'Plaus@10': '{:.3f}', 'Cohes@10': '{:.3f}', 'TagMatch@10': '{:.3f}',
}).highlight_max(
    subset=['R@10', 'R@10/ceil', 'MRR',
            'Cov@10', 'Nov@10', 'Div@10', 'Plaus@10', 'Cohes@10', 'TagMatch@10'],
    color='#d4edda', axis=0,
))

df3.to_csv(str(OUTPUT_DIR / 'baseline_comparison_minlen3.csv'), index=False)
print(f'Saved: {OUTPUT_DIR}/baseline_comparison_minlen3.csv')

In [ ]:
# Bar chart: R@10 and R@10/ceiling per method
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
order = list(METHODS.keys())
r_vals = [results_mlen3[m]['recall@10'] for m in order]
nr_vals = [results_mlen3[m]['normalized_recall@10'] for m in order]

colors_bar = ['#95a5a6' if 'unified' not in m else '#2980b9' for m in order]

ax1.barh(order, r_vals, color=colors_bar)
ax1.set_xlabel('Recall@10')
ax1.set_title('R@10 per method (min_seq=3)')
for i, v in enumerate(r_vals):
    ax1.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9)

ax2.barh(order, nr_vals, color=colors_bar)
ax2.axvline(x=1.0, color='black', linestyle=':', linewidth=1)
ax2.set_xlabel('Recall@10 / ceiling')
ax2.set_title(f'Normalized R@10 (ceiling={results_mlen3["popularity"]["ceiling_recall"]:.3f})')
for i, v in enumerate(nr_vals):
    ax2.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'baseline_recall_minlen3.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap of proxy vector per method
proxy_cols = ['Cov@10', 'Nov@10', 'Div@10', 'Plaus@10', 'Cohes@10', 'TagMatch@10']
proxy_df = df3.set_index('method')[proxy_cols]

fig, ax = plt.subplots(figsize=(10, len(proxy_df) * 0.45 + 1.5))
sns.heatmap(proxy_df, annot=True, fmt='.3f', cmap='YlGn',
            ax=ax, linewidths=0.5, vmin=0, vmax=1, cbar_kws={'label': 'metric value'})
ax.set_title('Proxy metrics per method (min_seq=3, top-10)')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'baseline_proxies_minlen3.png'), dpi=150, bbox_inches='tight')
plt.show()

## Reference: min_seq_length=5 (N≈17, wide CIs — do not over-interpret)

In [ ]:
results_mlen5 = {}
for name, fn in METHODS.items():
    results_mlen5[name] = evaluate_method(
        rec, fn, method_name=name,
        k_values=(5, 10, 20),
        protocol='trajectory_split', split_ratio=0.7,
        min_seq_length=5, proxy_k=10,
        verbose=False,
    )
    s = results_mlen5[name]
    print(f'  {name:22s}  N={s["n_evaluated"]:3d}  R@10={s["recall@10"]:.4f}  '
          f'MRR={s["mrr"]:.4f}')

rows5 = lift_table(results_mlen5,
                   baseline_keys=('popularity', 'category_popularity'), k=10)
df5 = pd.DataFrame(rows5)[display_cols]
print(f'\nCeiling (min_seq=5): {results_mlen5["popularity"]["ceiling_recall"]:.3f}')
display(df5.style.format({
    'R@10': '{:.4f}', 'R@10/ceil': '{:.3f}',
    'Δ vs popularity': '{:+.4f}', 'Δ vs category_popularity': '{:+.4f}',
    'MRR': '{:.4f}',
    'Cov@10': '{:.3f}', 'Nov@10': '{:.3f}', 'Div@10': '{:.3f}',
    'Plaus@10': '{:.3f}', 'Cohes@10': '{:.3f}', 'TagMatch@10': '{:.3f}',
}))
df5.to_csv(str(OUTPUT_DIR / 'baseline_comparison_minlen5.csv'), index=False)
print(f'Saved: {OUTPUT_DIR}/baseline_comparison_minlen5.csv')

## Interpretation checklist

Read the min_seq=3 table with these questions:

1. **Does unified beat popularity?** Look at `Δ vs popularity`. If <0.03 absolute, the nine-signal stack is not earning its complexity.
2. **Does unified beat category-popularity?** This is the stricter test — category-popularity already knows the trajectory's intent. If unified's Δ vs category-popularity is small, the gain is "knowing the dominant category," not the stack.
3. **Is Markov competitive?** If Markov's R@10 is close to unified, the flow graph alone captures most of the signal.
4. **Proxy tradeoffs**: popularity is high-coverage but low-novelty/diversity; proximity is high-cohesion but low-tag-match. Unified should sit above baselines on `Plaus@10`, `TagMatch@10`, and `Div@10` without collapsing `Cov@10`.
5. **Normalized recall**: `R@10/ceiling` > 0.7 means the method is near the structural ceiling; further recall gains require either more interactions per trajectory or cross-trajectory data.
